In [1]:
# 1. Import libraries and helper functions

import pandas as pd
from pathlib import Path
import os

def normalize_text(series):
    return series.astype(str).str.strip().str.title()

In [2]:
# 2. Path configuration

BASE_DIR = Path(os.environ.get("MASKING_DATA_PATH", "../data"))
RAW_DIR = BASE_DIR / "raw" / "us"
PROCESSED_DIR = BASE_DIR / "processed" / "us"

In [3]:
# 3. Load dataset

df = pd.read_excel(RAW_DIR / "returns_raw.xlsx").dropna(how='all')

In [4]:
# 4. Filter dataset columns

filtered_columns = [
    'CUSTOMER_ID', 'ITEM_ID', 'SALES_REP', 'INVOICE_QTY', 'RECEIPT_DATE', 'TAX_CODE', 'NET_TOTAL_VALUE'
]

df = df[filtered_columns].copy()

In [5]:
# 5. Data quality check

def data_quality_check(df, name="dataset", id_cols=None):
    """
    Runs a standard data quality check on any DataFrame.
    id_cols: str, list of str, or None. Pass the column(s) that should be
    unique keys - the function checks each one independently.
    """

    # 1. Header
    print(f"{'='*60}")
    print(f"DATA QUALITY CHECK: {name}")
    print(f"{'='*60}\n")

    # 2. Shape
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

    # 3. Data types
    print("--- Data types ---")
    print(df.dtypes)
    print()

    # 4. Nulls
    print("--- Null values ---")
    nulls = df.isna().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls found")
    print()

    # 5. Fully duplicated rows (entire row identical)
    print("--- Duplicated rows (entire row) ---")
    print(df.duplicated().sum())
    print()

    # 6. Key column(s) duplicated on their own, regardless of the rest of the row
    if id_cols:
        if isinstance(id_cols, str):
            id_cols = [id_cols]
        for col in id_cols:
            if col not in df.columns:
                print(f"WARNING: id_col '{col}' not found in this dataset - skipping\n")
                continue
            print(f"--- Duplicated '{col}' (key column) ---")
            id_dupes = df[col].duplicated().sum()
            print(id_dupes)
            if id_dupes > 0:
                print(f"WARNING: {id_dupes} duplicate '{col}' found - "
                      f"inspect whether other columns diverge between them:")
                print(df[df[col].duplicated(keep=False)].sort_values(col).head(20))
            print()

    # 7. Categorical columns: unique values + top values (spot inconsistent text)
    print("--- Categorical columns: unique value counts ---")
    for col in df.select_dtypes(include='object').columns:
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts().head(10))

    # 8. Numeric columns: descriptive stats (spot outliers, negatives, zeros)
    print("\n--- Numeric columns: descriptive stats ---")
    print(df.describe())

    print(f"\n{'='*60}\n")
    
data_quality_check(df, name="returns_raw", id_cols=None)

DATA QUALITY CHECK: returns_raw

Shape: 39740 rows x 7 columns

--- Data types ---
CUSTOMER_ID                 int64
ITEM_ID                     int64
SALES_REP                  object
INVOICE_QTY                 int64
RECEIPT_DATE       datetime64[ns]
TAX_CODE                   object
NET_TOTAL_VALUE           float64
dtype: object

--- Null values ---
No nulls found

--- Duplicated rows (entire row) ---
148

--- Categorical columns: unique value counts ---

SALES_REP: 25 unique values
SALES_REP
Angie Henderson    1728
Jamie Arnold       1678
Patty Perez        1642
Tommy Walter       1639
Daniel Wagner      1627
Cristian Santos    1625
Carla Gray         1623
Ethan Adams        1618
Holly Wood         1611
Monica Herrera     1602
Name: count, dtype: int64

TAX_CODE: 8 unique values
TAX_CODE
TX_PUR_1101     5061
TX_MISC_1999    5004
TX_RET_1202     4988
TX_ADJ_1949     4976
TX_CAN_1949     4958
TX_RET_1201     4946
TX_RET_2202     4910
TX_PUR_1102     4897
Name: count, dtype: int64

-

In [6]:
# 6. Rename columns

df.columns = df.columns.str.lower()

In [7]:
# 7. Validate duplicated rows

# returns_raw has no order_id, so there's no natural single-transaction
# key like in sales. The closest equivalent to a unique return event is
# the combination of customer_id + item_id + receipt_date - if these
# three match, it's very likely the same return recorded more than once.

composite_dupes = df.duplicated(subset=['customer_id', 'item_id', 'receipt_date']).sum()
print(f"{composite_dupes} customer_id + item_id + receipt_date combinations duplicated")

148 customer_id + item_id + receipt_date combinations duplicated


In [8]:
# 8. Investigate whether these are real duplicates or legitimate repeats

# Same customer_id + item_id + receipt_date can, in principle, repeat if
# a customer returns the same product on the same day in separate
# transactions. Only rows identical across EVERY column represent an
# actual duplicate entry.

has_valid_date = df['receipt_date'].notna()

real_dupes = df[has_valid_date & df.duplicated(subset=['customer_id', 'item_id', 'receipt_date'], keep=False)].sort_values(['customer_id', 'item_id'])
print(real_dupes[['customer_id', 'item_id', 'receipt_date', 'invoice_qty', 'net_total_value']].head(5))

       customer_id  item_id receipt_date  invoice_qty  net_total_value
8               36      806   2024-01-24           21           618.76
31144           36      806   2024-01-24           21           618.76
53              41      864   2022-11-21           24          1459.05
12984           41      864   2022-11-21           24          1459.05
146             42     8047   2021-11-26           15          1422.62


In [9]:
# 9. Remove fully duplicated rows

# A fully identical order line (same customer_id, item_id, date, quantity, value,
# and tax_code) should never occur - if found, it indicates a
# duplicate entry from the source system or the data generation process.
# This check is kept as a standing safeguard, even when no duplicates are
# found in a given run.

before = len(df)
df = df.drop_duplicates(keep='first')
after = len(df)

print(f"{before - after} fully duplicated rows removed")

148 fully duplicated rows removed


In [10]:
# 10. Final check

assert df.duplicated().sum() == 0, "Fully duplicated rows still remain!"

In [11]:
# 11. Replace sales_rep name with rep_id from the dimension table

df = df.copy()

reps_dim = pd.read_csv(PROCESSED_DIR / "dim_reps.csv")

df['sales_rep'] = normalize_text(df['sales_rep'])

df = df.merge(
    reps_dim[['rep_id', 'rep_name']],
    left_on='sales_rep',
    right_on='rep_name',
    how='left'
)

# Validate: every sales_rep name must have matched a rep_id
assert df['rep_id'].isna().sum() == 0, "Some sales_rep names did not match any rep_id!"

df = df.drop(columns=['sales_rep', 'rep_name'])

# Insert rep_id next to others ids columns
df = df[['customer_id', 'item_id', 'rep_id', 'invoice_qty', 'receipt_date', 'tax_code', 'net_total_value']]

In [12]:
# 14. Convert data types

df['customer_id'] = df['customer_id'].astype(str)
df['item_id'] = df['item_id'].astype(str)
df['rep_id'] = df['rep_id'].astype(str)
df['tax_code'] = df['tax_code'].astype('category')

In [13]:
# 15. Sales return business logic layer

# Create a flag to identify sales returns:
# - Sales returns occur when a customer sends a product back, reversing a previous sale
# - We use transaction codes that represent this type of operation (TX_RET_120* and TX_RET_220* prefixes)

df['is_sales_return'] = df['tax_code'].str.startswith(('TX_RET_120', 'TX_RET_220'), na=False)

# Calculate the return value for analysis and dashboards
# (0 for transactions that aren't a real sales return)

df['return_value'] = df['net_total_value'].where(df['is_sales_return'], 0.0)

In [14]:
# 16. Final checks

assert df.duplicated().sum() == 0, "Fully duplicated rows still remain!"
#assert df['item_id'].astype(str).isin(products_dim['item_id'].astype(str)).all(), "Orphan ITEM_ID found!"
assert df['rep_id'].isna().sum() == 0, "Missing rep_id found!"
#assert df['customer_id'].isin(customers_dim['cust_id'].astype(int)).all(), "Orphan CUSTOMER_ID found!"

print(f"Success! {len(df)} return records validated.")
print(f"{df['is_sales_return'].sum()} rows flagged as sales returns")
print(f"Total return value: ${df['return_value'].sum():,.2f}")

Success! 39592 return records validated.
14794 rows flagged as sales returns
Total return value: $18,848,468.96


In [15]:
# 17. Export cleaned dataset for BI 

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "fact_returns.csv", index=False)

print(f"Success! {len(df)} sales records exported.")

Success! 39592 sales records exported.
